# Tutorial Part 6: Introduction to Graph Convolutions

In this tutorial we will learn more about "graph convolutions." These are one of the most powerful deep learning tools for working with molecular data. The reason for this is that molecules can be naturally viewed as graphs.

![Molecular Graph](https://github.com/deepchem/deepchem/blob/master/examples/tutorials/basic_graphs.gif?raw=1)

Note how standard chemical diagrams of the sort we're used to from high school lend themselves naturally to visualizing molecules as graphs. In the remainder of this tutorial, we'll dig into this relationship in significantly more detail. This will let us get a deeper understanding of how these systems work.

## Colab

This tutorial and the rest in this sequence are designed to be done in Google colab. If you'd like to open this notebook in colab, you can use the following link.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deepchem/deepchem/blob/master/examples/tutorials/06_Introduction_to_Graph_Convolutions.ipynb)

## Setup

To run DeepChem within Colab, run the install cell below. It usually takes a minute or two. If Colab asks you to restart the session afterwards, do so (**Runtime → Restart session**) and continue with the next cell; there is no need to run the install cell again. You can of course run this tutorial locally if you prefer; in that case install the same packages in your own Python environment (e.g. `pip install deepchem rdkit tf_keras tensorflow`) instead of running the install cell.

In [1]:
# DeepChem itself, RDKit (reads molecules) and tf_keras. DeepChem's TensorFlow models and layers
# need the legacy Keras 2 API, which recent TensorFlow versions only provide through "tf_keras".
%pip install -q deepchem rdkit tf_keras

# Colab normally ships with TensorFlow; install it only if it is missing.
import importlib.util
if importlib.util.find_spec("tensorflow") is None:
    print("TensorFlow not found - installing it ...")
    %pip install -q tensorflow

Now we import `deepchem`. Two details about the import cell below:

* It sets the environment variable `TF_USE_LEGACY_KERAS=1` so that TensorFlow uses Keras 2, which is what DeepChem's TensorFlow models and graph layers expect. This **must happen before TensorFlow or DeepChem is imported**. If you ever see an error about `keras.optimizers.legacy` or Keras 3, restart the session (**Runtime → Restart session**) and run the import cell first.
* We silence a few harmless warnings (deprecation notices and repeated RDKit messages) so that the real output stays readable.

In [2]:
import os, sys, warnings

# Use Keras 2 (tf_keras) inside TensorFlow. Must be set BEFORE importing tensorflow/deepchem.
os.environ["TF_USE_LEGACY_KERAS"] = "1"
if "tensorflow" in sys.modules:
    print("WARNING: TensorFlow was already imported in this session. "
          "Use Runtime > Restart session and run this cell before any other TensorFlow code.")

# Silence harmless warnings (set BEFORE importing deepchem):
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
warnings.filterwarnings("ignore", message=".*pkg_resources.*")
warnings.filterwarnings("ignore", message=".*invalid escape sequence.*", category=SyntaxWarning)
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

import tensorflow as tf
tf.get_logger().setLevel("ERROR")     # quiet TensorFlow's deprecation notices, also while DeepChem is imported
import deepchem as dc

print("DeepChem", dc.__version__, "| TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none (CPU works too, just slower)")

# DeepChem silently leaves out its TensorFlow models if TensorFlow cannot be imported;
# fail early with a clear message instead of a confusing error later.
if not hasattr(dc.models, "GraphConvModel"):
    raise ImportError("dc.models.GraphConvModel is unavailable: DeepChem could not load its TensorFlow "
                      "models. Check the messages above, restart the session and re-run the install and import cells.")

wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.
wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.


DeepChem 2.5.0 | TensorFlow 2.21.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


# What are Graph Convolutions?

Consider a standard convolutional neural network (CNN) of the sort commonly used to process images.  The input is a grid of pixels.  There is a vector of data values for each pixel, for example the red, green, and blue color channels.  The data passes through a series of convolutional layers.  Each layer combines the data from a pixel and its neighbors to produce a new data vector for the pixel.  Early layers detect small scale local patterns, while later layers detect larger, more abstract patterns.  Often the convolutional layers alternate with pooling layers that perform some operation such as max or min over local regions.

Graph convolutions are similar, but they operate on a graph.  They begin with a data vector for each node of the graph (for example, the chemical properties of the atom that node represents).  Convolutional and pooling layers combine information from connected nodes (for example, atoms that are bonded to each other) to produce a new data vector for each node.

# Training a GraphConvModel

Let's use the MoleculeNet suite to load the Tox21 dataset. To featurize the data in a way that graph convolutional networks can use, we use DeepChem's `ConvMol` featurizer (the one behind `featurizer='GraphConv'`), wrapped so that it copes with molecules that fail to parse (see the next cell). The MoleculeNet call returns a training set, a validation set, and a test set for us to use. It also returns `tasks`, a list of the task names, and `transformers`, a list of data transformations that were applied to preprocess the dataset. (Most deep networks are quite finicky and require a set of data transformations to ensure that training proceeds stably.)

**A small robustness wrapper.** Normally we would just pass `featurizer='GraphConv'`. However, the current version of RDKit can no longer parse a handful of the Tox21 SMILES strings (8 of 7,831 in our test run). When a molecule fails, DeepChem 2.5.0 (the version installed in our Colab test) stores an empty placeholder and then tries to pack all the results into one NumPy array. Modern NumPy refuses to do that for entries of different shapes, and loading stops with `ValueError: ... inhomogeneous shape`.

The wrapper below produces exactly the same `ConvMol` features, but stores the results in an *object* array, so the failed molecules simply end up as empty placeholders, which DeepChem's data loader then drops. We lose a few molecules out of thousands.

In [3]:
import numpy as np
from rdkit import Chem

class RobustConvMolFeaturizer(dc.feat.ConvMolFeaturizer):
  """Same ConvMol features as featurizer='GraphConv', but tolerant of molecules that fail to parse.

  Failed molecules become empty arrays, which DeepChem's data loader removes afterwards.
  """
  def featurize(self, molecules, log_every_n=1000):
    if isinstance(molecules, (str, Chem.Mol)):
      molecules = [molecules]
    else:
      molecules = list(molecules)
    features = np.empty(len(molecules), dtype=object)    # object array: entries may differ in type/shape
    n_failed = 0
    for i, mol in enumerate(molecules):
      try:
        if isinstance(mol, str):
          mol = Chem.MolFromSmiles(mol)       # returns None if RDKit cannot parse the SMILES
        features[i] = self._featurize(mol)    # raises for None, handled below
      except Exception:
        features[i] = np.array([])            # empty array = "this molecule failed"
        n_failed += 1
    if n_failed:
      print(f"Note: {n_failed} of {len(molecules)} molecules could not be featurized and will be skipped.")
    return features

In [4]:
tasks, datasets, transformers = dc.molnet.load_tox21(featurizer=RobustConvMolFeaturizer())
train_dataset, valid_dataset, test_dataset = datasets

Note: 8 of 7831 molecules could not be featurized and will be skipped.


Let's now train a graph convolutional network on this dataset. DeepChem has the class `GraphConvModel` that wraps a standard graph convolutional architecture underneath the hood for user convenience. Let's instantiate an object of this class and train it on our dataset.

In [5]:
n_tasks = len(tasks)
model = dc.models.GraphConvModel(n_tasks, mode='classification')
model.fit(train_dataset, nb_epoch=50)

0.2680793762207031

Let's try to evaluate the performance of the model we've trained. For this, we need to define a metric, a measure of model performance. `dc.metrics` holds a collection of metrics already. For this dataset, it is standard to use the ROC-AUC score, the area under the receiver operating characteristic curve (which summarises how well the model ranks positive examples above negative ones, across all decision thresholds). Luckily, the ROC-AUC score is already available in DeepChem.

To measure the performance of the model under this metric, we can use the convenience function `model.evaluate()`.

In [6]:
metric = dc.metrics.Metric(dc.metrics.roc_auc_score)
print('Training set score:', model.evaluate(train_dataset, [metric], transformers))
print('Test set score:', model.evaluate(test_dataset, [metric], transformers))

Training set score: {'roc_auc_score': np.float64(0.9720317580236971)}
Test set score: {'roc_auc_score': np.float64(0.6884821155482511)}


The results are pretty good, and `GraphConvModel` is very easy to use. But what's going on under the hood? Could we build GraphConvModel ourselves? Of course! DeepChem provides Keras layers for all the calculations involved in a graph convolution. We are going to apply the following layers from DeepChem.

-  `GraphConv` layer: This layer implements the graph convolution. The graph convolution combines per-node feature vectures in a nonlinear fashion with the feature vectors for neighboring nodes.  This "blends" information in local neighborhoods of a graph.

- `GraphPool` layer: This layer does a max-pooling over the feature vectors of atoms in a neighborhood. You can think of this layer as analogous to a max-pooling layer for 2D convolutions but which operates on graphs instead.

- `GraphGather`: Many graph convolutional networks manipulate feature vectors per graph-node. For a molecule for example, each node might represent an atom, and the network would manipulate atomic feature vectors that summarize the local chemistry of the atom. However, at the end of the application, we will likely want to work with a molecule level feature representation. This layer creates a graph level feature vector by combining all the node-level feature vectors.

Apart from this we are going to apply standard neural network layers such as [Dense](https://keras.io/api/layers/core_layers/dense/), [BatchNormalization](https://keras.io/api/layers/normalization_layers/batch_normalization/) and [Softmax](https://keras.io/api/layers/activation_layers/softmax/) layer.

In [7]:
from deepchem.models.layers import GraphConv, GraphPool, GraphGather
import tensorflow as tf
layers = tf.keras.layers      # (with TF_USE_LEGACY_KERAS=1 this is the Keras 2 layers module)

batch_size = 100

class MyGraphConvModel(tf.keras.Model):

  def __init__(self):
    super(MyGraphConvModel, self).__init__()
    self.gc1 = GraphConv(128, activation_fn=tf.nn.tanh)
    self.batch_norm1 = layers.BatchNormalization()
    self.gp1 = GraphPool()

    self.gc2 = GraphConv(128, activation_fn=tf.nn.tanh)
    self.batch_norm2 = layers.BatchNormalization()
    self.gp2 = GraphPool()

    self.dense1 = layers.Dense(256, activation=tf.nn.tanh)
    self.batch_norm3 = layers.BatchNormalization()
    self.readout = GraphGather(batch_size=batch_size, activation_fn=tf.nn.tanh)

    self.dense2 = layers.Dense(n_tasks*2)
    self.logits = layers.Reshape((n_tasks, 2))
    self.softmax = layers.Softmax()

  def call(self, inputs):
    gc1_output = self.gc1(inputs)
    batch_norm1_output = self.batch_norm1(gc1_output)
    gp1_output = self.gp1([batch_norm1_output] + inputs[1:])

    gc2_output = self.gc2([gp1_output] + inputs[1:])
    batch_norm2_output = self.batch_norm2(gc2_output)      # second block gets its OWN batch-norm layer
    gp2_output = self.gp2([batch_norm2_output] + inputs[1:])

    dense1_output = self.dense1(gp2_output)
    batch_norm3_output = self.batch_norm3(dense1_output)
    readout_output = self.readout([batch_norm3_output] + inputs[1:])

    logits_output = self.logits(self.dense2(readout_output))
    return self.softmax(logits_output)

We can now see more clearly what is happening.  There are two convolutional blocks, each consisting of a `GraphConv`, followed by batch normalization, followed by a `GraphPool` to do max pooling.  We finish up with a dense layer, another batch normalization, a `GraphGather` to combine the data from all the different nodes, and a final dense layer to produce the global output.

Let's now create the DeepChem model which will be a wrapper around the Keras model that we just created. We will also specify the loss function so the model know the objective to minimize.

In [8]:
model = dc.models.KerasModel(MyGraphConvModel(), loss=dc.models.losses.CategoricalCrossEntropy())

What are the inputs to this model?  A graph convolution requires a complete description of each molecule, including the list of nodes (atoms) and a description of which ones are bonded to each other.  In fact, if we inspect the dataset we see that the feature array contains Python objects of type `ConvMol`.

In [9]:
test_dataset.X[0]

Models expect arrays of numbers as their inputs, not Python objects.  We must convert the `ConvMol` objects into the particular set of arrays expected by the `GraphConv`, `GraphPool`, and `GraphGather` layers.  Fortunately, the `ConvMol` class includes the code to do this, as well as to combine all the molecules in a batch to create a single set of arrays.

The following code creates a Python generator that given a batch of data generates the lists of inputs, labels, and weights whose values are Numpy arrays. `atom_features` holds a feature vector of length 75 for each atom. The other inputs are required to support minibatching in TensorFlow. `degree_slice` is an indexing convenience that makes it easy to locate atoms from all molecules with a given degree. `membership` determines the membership of atoms in molecules (atom `i` belongs to molecule `membership[i]`). `deg_adjs` is a list that contains adjacency lists grouped by atom degree. For more details, check out the [code](https://github.com/deepchem/deepchem/blob/master/deepchem/feat/mol_graphs.py).

In [10]:
from deepchem.metrics import to_one_hot
from deepchem.feat.mol_graphs import ConvMol
import numpy as np

def data_generator(dataset, epochs=1):
  for ind, (X_b, y_b, w_b, ids_b) in enumerate(dataset.iterbatches(batch_size, epochs,
                                                                   deterministic=False, pad_batches=True)):
    multiConvMol = ConvMol.agglomerate_mols(X_b)
    inputs = [multiConvMol.get_atom_features(), multiConvMol.deg_slice, np.array(multiConvMol.membership)]
    for i in range(1, len(multiConvMol.get_deg_adjacency_lists())):
      inputs.append(multiConvMol.get_deg_adjacency_lists()[i])
    labels = [to_one_hot(y_b.flatten(), 2).reshape(-1, n_tasks, 2)]
    weights = [w_b]
    yield (inputs, labels, weights)

Now, we can train the model using `fit_generator(generator)` which will use the generator we've defined to train the model.

In [11]:
model.fit_generator(data_generator(train_dataset, epochs=50))

0.2426454734802246

Now that we have trained our graph convolutional method, let's evaluate its performance. We again have to use our defined generator to evaluate model performance.

In [12]:
print('Training set score:', model.evaluate_generator(data_generator(train_dataset), [metric], transformers))
print('Test set score:', model.evaluate_generator(data_generator(test_dataset), [metric], transformers))

Training set score: {'roc_auc_score': np.float64(0.9742907958171226)}
Test set score: {'roc_auc_score': np.float64(0.6921446898819202)}


Success! The model we've constructed is a working graph convolutional network, and it reaches a broadly similar level of performance to `GraphConvModel`. The scores will not be identical: the layer sizes differ from those of `GraphConvModel`, and training starts from random weights, so your numbers will vary from run to run. If you're looking to build your own custom models, you can follow the example we've provided here to do so. We hope to see exciting constructions from your end soon!

# Congratulations! Time to join the Community!

Congratulations on completing this tutorial notebook! If you enjoyed working through the tutorial, and want to continue working with DeepChem, we encourage you to finish the rest of the tutorials in this series. You can also help the DeepChem community in the following ways:

## Star DeepChem on [GitHub](https://github.com/deepchem/deepchem)
This helps build awareness of the DeepChem project and the tools for open source drug discovery that we're trying to build.

## Join the DeepChem Discord
The DeepChem [Discord](https://discord.gg/cGzwCdrUqS) hosts a number of scientists, developers, and enthusiasts interested in deep learning for the life sciences. Join the conversation!

# Troubleshooting

| Symptom | What to do |
|---|---|
| `ModuleNotFoundError` for `deepchem`, or odd NumPy / binary-compatibility errors right after the install cell | **Runtime → Restart session**, then run the import cell (no need to reinstall). |
| `ImportError: keras.optimizers.legacy is not supported in Keras 3` | TensorFlow was imported before `TF_USE_LEGACY_KERAS` was set, or `tf_keras` is missing. Restart the session, make sure the install cell ran, and run the import cell first. |
| `AttributeError: module 'deepchem.models' has no attribute 'GraphConvModel'` | DeepChem could not load TensorFlow. Read the warnings printed by the import cell; `%pip install tensorflow tf_keras`, restart, and try again. |
| `ValueError: setting an array element with a sequence ... inhomogeneous shape` while loading Tox21 | A molecule failed to featurize. Make sure the `RobustConvMolFeaturizer` cell ran and that `load_tox21` is called with `featurizer=RobustConvMolFeaturizer()`. |\n| `ImportError` for `GraphConv`, `GraphPool` or `GraphGather` from `deepchem.models.layers` | Your DeepChem version is organised differently. Check `dc.__version__` (printed by the import cell) and the DeepChem documentation for that version. |
| Training takes long | Fitting both models for 50 epochs can take several minutes, more on CPU. Use **Runtime → Change runtime type** to pick a GPU, or lower `nb_epoch` for a quick test. |